In [2]:
import pandas as pd
import numpy as np
import nltk
import torch
import torch.nn.functional as F

from gensim.models import CoherenceModel, LdaModel
from gensim.corpora.dictionary import Dictionary
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

# -- NLTK İndirmeleri --
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

########################################################
# 1) Yardımcı Fonksiyonlar: Cümle Bölme ve Tokenize
########################################################

def sentence_split(text):
    """Metni nltk.sent_tokenize ile cümlelere böler."""
    from nltk.tokenize import sent_tokenize
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]

def tokenize_text(text):
    """Basit tokenize işlemi."""
    return nltk.word_tokenize(text)

########################################################
# 2) Global LDA: Tüm Cümleler Üzerinde Otomatik Aspect Çıkarımı
########################################################

def find_optimal_lda(tokenized_texts, start=2, limit=10, step=1, passes=10):
    """
    Tüm dataset (cümle bazlı) için, (start..limit) aralığında topic sayıları dener,
    coherence'e göre en iyi modeli döndürür.
    """
    dictionary = Dictionary(tokenized_texts)
    corpus = [dictionary.doc2bow(t) for t in tokenized_texts]
    
    best_coherence = -999
    best_model = None
    best_num_topics = None
    
    print(f"\n[find_optimal_lda] Tüm veride topic sayısı aranıyor: {start}..{limit} arası")
    for num_topics in range(start, limit, step):
        tmp_model = LdaModel(corpus=corpus,
                             id2word=dictionary,
                             num_topics=num_topics,
                             passes=passes,
                             random_state=42)
        cm = CoherenceModel(model=tmp_model,
                            texts=tokenized_texts,
                            dictionary=dictionary,
                            coherence='c_v')
        coh_val = cm.get_coherence()
        print(f"   {num_topics} topics => coherence={coh_val:.4f}")
        if coh_val > best_coherence:
            best_coherence = coh_val
            best_model = tmp_model
            best_num_topics = num_topics
    
    print(f"\n[find_optimal_lda] En iyi topic sayısı={best_num_topics}, coherence={best_coherence:.4f}")
    return best_num_topics, best_model, dictionary, corpus

def get_topic_keywords(lda_model, num_words=8):
    """
    Her topic için en iyi num_words kelimeyi döndürür.
    """
    topic_keywords = {}
    for t_id, kw_list in lda_model.show_topics(num_topics=-1, num_words=num_words, formatted=False):
        topic_keywords[t_id] = [term for term, _ in kw_list]
    return topic_keywords

########################################################
# 3) Aspect-Based Sentiment Model (ABSA) Kullanımı
########################################################

# Global aspect-based sentiment modeli: yangheng/deberta-v3-base-absa-v1.1
absa_model_name = "yangheng/deberta-v3-base-absa-v1.1"
absa_tokenizer = AutoTokenizer.from_pretrained(absa_model_name, use_fast=False)
absa_model = AutoModelForSequenceClassification.from_pretrained(absa_model_name)
absa_model.eval()

def absa_sentiment(text, aspect):
    """
    (text, aspect) çiftini kullanarak aspect bazlı sentiment analizi yapar.
    Dönen skor:
      - '1 star'/'2 star' => -1 (negative)
      - '3 star' => 0 (neutral)
      - '4 star'/'5 star' => +1 (positive)
    """
    inputs = absa_tokenizer(text, aspect, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = absa_model(**inputs)
    probs = F.softmax(outputs.logits[0], dim=-1)
    label_id = torch.argmax(probs).item()
    label = absa_model.config.id2label[label_id]
    if label.lower().startswith("1 star") or label.lower().startswith("2 star"):
        score = -1
    elif label.lower().startswith("3 star"):
        score = 0
    else:
        score = 1
    return score, score_to_label(score)

def score_to_label(score):
    """Skor -1,0,+1'den duygu etiketini verir."""
    if score < 0:
        return "negative"
    elif score == 0:
        return "neutral"
    else:
        return "positive"

########################################################
# 4) Doküman (Cümle) İçin Dominant Topic
########################################################

def get_dominant_topics(lda_model, bow, threshold=0.2):
    """
    Bir cümlenin (dokümanın) LDA topic dağılımını hesaplar ve threshold üstü topic ID'lerini döndürür.
    """
    topic_weights = lda_model.get_document_topics(bow)
    return [t_id for t_id, w in topic_weights if w >= threshold]

########################################################
# 5) Otel Bazında PCA (Opsiyonel)
########################################################

def pca_for_hotel(hotel_reviews):
    """
    Otel bazında, tüm yorumları TF-IDF + PCA ile özetler.
    """
    if len(hotel_reviews) < 2:
        return (0.0, 0.0), (0.0, 0.0)
    vectorizer = TfidfVectorizer(token_pattern=r'\S+')
    X = vectorizer.fit_transform(hotel_reviews)
    if X.shape[0] < 2:
        return (0.0, 0.0), (0.0, 0.0)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X.toarray())
    avg_pca = np.mean(X_pca, axis=0)
    var_ratio = pca.explained_variance_ratio_
    return avg_pca, var_ratio

########################################################
# 6) Ana Kod: Global LDA + Cümle Bazlı Aspect-Based Sentiment
########################################################

if __name__ == "__main__":
    print("[INFO] CSV verisi okunuyor...")
    df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8")
    df.dropna(subset=["hotel_name", "processed_final_review"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"[INFO] Toplam {df.shape[0]} yorum okundu.")

    # 6.1) Tüm yorumları cümlelere böl (global corpus oluşturmak için)
    all_sentences = []
    sentence_records = []
    for idx, row in df.iterrows():
        hotel = row["hotel_name"]
        full_text = row["processed_final_review"]
        sentences = sentence_split(full_text)
        for sent in sentences:
            all_sentences.append(tokenize_text(sent))
            sentence_records.append({
                "hotel_name": hotel,
                "sentence": sent
            })
        if idx % 100 == 0:
            print(f"[INFO] {idx} yorum cümlelerine ayrıldı...")
    sentences_df = pd.DataFrame(sentence_records)
    print(f"[INFO] Toplam {sentences_df.shape[0]} cümle elde edildi.")

    # 6.2) Global LDA eğitimi: Tüm cümleler üzerinden
    print("\n[INFO] Global LDA eğitiliyor (cümle bazlı)...")
    best_topic_num, lda_model, dictionary, corpus = find_optimal_lda(
        tokenized_texts=all_sentences,
        start=2,
        limit=10,
        step=1,
        passes=10
    )
    print(f"[INFO] Global LDA ile en iyi topic sayısı: {best_topic_num}")

    # LDA'dan çıkan topic kelimeleri (aspect tanımı olarak kullanılacak)
    topic_keywords = get_topic_keywords(lda_model, num_words=8)
    for t_id, words in topic_keywords.items():
        print(f"   Topic {t_id}: {words}")

    # 6.3) Her cümle için, global LDA'dan çıkarılan konulara göre ABSA uygulayalım
    results = []
    total_sentences = sentences_df.shape[0]
    for idx, row in sentences_df.iterrows():
        hotel = row["hotel_name"]
        sent = row["sentence"]
        tokens = tokenize_text(sent)
        bow = dictionary.doc2bow(tokens)
        # Hangi topic'lere ait?
        dominant_topics = get_dominant_topics(lda_model, bow, threshold=0.2)
        # Eğer cümle herhangi bir topic'e aitse, her biri için ABSA çalıştır
        for t_id in dominant_topics:
            aspect = " ".join(topic_keywords.get(t_id, []))  # Topic kelimeleri aspect olarak kullanılacak
            score, label = absa_sentiment(sent, aspect)
            results.append({
                "hotel_name": hotel,
                "sentence": sent,
                "topic_id": t_id,
                "aspect_keywords": aspect,
                "sentiment_score": score,
                "sentiment_label": label
            })
        if idx % 200 == 0:
            print(f"[INFO] {idx}/{total_sentences} cümle işlendi...")
    absa_df = pd.DataFrame(results)
    print(f"[INFO] ABSA sonuçları elde edildi: {absa_df.shape[0]} satır.")

    # 6.4) Otel bazında PCA (isteğe bağlı)
    print("\n[INFO] Otel bazında PCA hesaplanıyor...")
    pca_records = []
    for hotel in df["hotel_name"].unique():
        hotel_reviews = df[df["hotel_name"] == hotel]["processed_final_review"].tolist()
        avg_pca, var_ratio = pca_for_hotel(hotel_reviews)
        pca_records.append({
            "hotel_name": hotel,
            "pca_pc1": float(avg_pca[0]),
            "pca_pc2": float(avg_pca[1]),
            "var_ratio_pc1": float(var_ratio[0]) if len(var_ratio)>0 else 0.0,
            "var_ratio_pc2": float(var_ratio[1]) if len(var_ratio)>1 else 0.0
        })
    pca_df = pd.DataFrame(pca_records)
    print(f"[INFO] PCA sonuçları: {pca_df.shape[0]} otel için özet çıkarıldı.")

    # 6.5) CSV Çıktıları: Cümle bazlı ABSA ve Otel bazlı özet pivot
    print("\n[INFO] CSV dosyaları oluşturuluyor...")
    absa_df.to_csv("absa_sentences.csv", index=False, encoding="utf-8-sig")
    pca_df.to_csv("hotel_pca_summary.csv", index=False, encoding="utf-8-sig")
    print("[INFO] 'absa_sentences.csv' (cümle bazlı ABSA) ve 'hotel_pca_summary.csv' (otel bazında PCA) kaydedildi.")
    
    # Ek: Otel & Topic bazında ortalama sentiment pivot
    pivot_df = absa_df.groupby(["hotel_name", "topic_id"]).agg({
        "sentiment_score": "mean"
    }).reset_index().rename(columns={"sentiment_score": "avg_sentiment"})
    pivot_df["avg_sentiment_label"] = pivot_df["avg_sentiment"].apply(score_to_label)
    pivot_df.to_csv("hotel_topic_avg_sentiment.csv", index=False, encoding="utf-8-sig")
    print("[INFO] 'hotel_topic_avg_sentiment.csv' (otel & topic pivot) kaydedildi.")
    
    print("\n***** Tüm işlemler tamamlandı! *****")


[INFO] CSV verisi okunuyor...
[INFO] Toplam 1271 yorum okundu.
[INFO] 0 yorum cümlelerine ayrıldı...
[INFO] 100 yorum cümlelerine ayrıldı...
[INFO] 200 yorum cümlelerine ayrıldı...
[INFO] 300 yorum cümlelerine ayrıldı...
[INFO] 400 yorum cümlelerine ayrıldı...
[INFO] 500 yorum cümlelerine ayrıldı...
[INFO] 600 yorum cümlelerine ayrıldı...
[INFO] 700 yorum cümlelerine ayrıldı...
[INFO] 800 yorum cümlelerine ayrıldı...
[INFO] 900 yorum cümlelerine ayrıldı...
[INFO] 1000 yorum cümlelerine ayrıldı...
[INFO] 1100 yorum cümlelerine ayrıldı...
[INFO] 1200 yorum cümlelerine ayrıldı...
[INFO] Toplam 1271 cümle elde edildi.

[INFO] Global LDA eğitiliyor (cümle bazlı)...

[find_optimal_lda] Tüm veride topic sayısı aranıyor: 2..10 arası
   2 topics => coherence=0.3604
   3 topics => coherence=0.4039
   4 topics => coherence=0.3936
   5 topics => coherence=0.3905
   6 topics => coherence=0.3770
   7 topics => coherence=0.3763
   8 topics => coherence=0.3644
   9 topics => coherence=0.3624

[find_op

C:\Users\catsu\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\decomposition\_pca.py:559: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio_ = explained_variance_ / total_var


In [1]:
import pandas as pd
import numpy as np
import nltk
from gensim.models import LdaModel, CoherenceModel
from gensim.corpora.dictionary import Dictionary
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# NLTK İndirmeleri
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

#######################################
# Fonksiyonlar
#######################################

def preprocess_text(text):
    """Metni küçük harfe çevirip tokenize eder."""
    return nltk.word_tokenize(text.lower())

def read_reviews(csv_path, text_col="processed_final_review"):
    """
    CSV’den, text_col sütunundaki veriyi okur.
    """
    df = pd.read_csv(csv_path, encoding="utf-8")
    df.dropna(subset=["hotel_name", text_col], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df[text_col].tolist()

#######################################
# TF-IDF Keyword Extraction
#######################################

def extract_tfidf_keywords(texts, top_n=10):
    """
    TF-IDF matrisini oluşturur ve tüm belgelerde toplam TF-IDF değerine göre 
    en yüksek top_n kelimeyi ve değerlerini döndürür.
    """
    vectorizer = TfidfVectorizer(stop_words="english", token_pattern=r'\S+')
    X_tfidf = vectorizer.fit_transform(texts)
    feature_names = np.array(vectorizer.get_feature_names_out())
    tfidf_sum = np.array(X_tfidf.sum(axis=0)).flatten()
    top_indices = tfidf_sum.argsort()[-top_n:][::-1]
    top_keywords = feature_names[top_indices]
    top_scores = tfidf_sum[top_indices]
    return list(zip(top_keywords, top_scores))

#######################################
# LDA Keyword Extraction
#######################################

def find_optimal_lda(tokenized_texts, start=2, limit=10, step=1, passes=10):
    """
    Tüm dataset için (tokenized_texts) optimal topic sayısını (coherence'e göre) bulur.
    """
    dictionary = Dictionary(tokenized_texts)
    corpus = [dictionary.doc2bow(t) for t in tokenized_texts]
    
    best_coherence = -999
    best_model = None
    best_num_topics = None
    
    print(f"\n[INFO] Optimal topic sayısı aranıyor: {start}..{limit} arası")
    for num_topics in range(start, limit, step):
        tmp_model = LdaModel(corpus=corpus,
                             id2word=dictionary,
                             num_topics=num_topics,
                             passes=passes,
                             random_state=42)
        cm = CoherenceModel(model=tmp_model,
                            texts=tokenized_texts,
                            dictionary=dictionary,
                            coherence='c_v')
        coh_val = cm.get_coherence()
        print(f"   {num_topics} topics => coherence={coh_val:.4f}")
        if coh_val > best_coherence:
            best_coherence = coh_val
            best_model = tmp_model
            best_num_topics = num_topics
    
    print(f"\n[INFO] En iyi topic sayısı = {best_num_topics}, coherence = {best_coherence:.4f}")
    return best_num_topics, best_model, dictionary, corpus

def extract_lda_keywords(lda_model, top_n=10):
    """
    Her topic için top_n kelime ve ağırlıklarını döndürür.
    """
    topic_keywords = {}
    for topic_id in range(lda_model.num_topics):
        terms = lda_model.show_topic(topic_id, topn=top_n)
        topic_keywords[topic_id] = [(term, weight) for term, weight in terms]
    return topic_keywords

#######################################
# Ana Çalışma
#######################################

if __name__ == "__main__":
    # CSV'den veriyi oku (dosya yolunu kendi ortamınıza göre düzenleyin)
    csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
    texts = read_reviews(csv_path, text_col="processed_final_review")
    print(f"[INFO] CSV'den {len(texts)} yorum okundu.")

    # TF-IDF Keyword Extraction
    tfidf_top_keywords = extract_tfidf_keywords(texts, top_n=10)
    print("\n==== TF-IDF Keyword Extraction (Top 10) ====")
    for word, score in tfidf_top_keywords:
        print(f"{word:15} {score:.4f}")

    # LDA Keyword Extraction
    # Tokenize tüm metinler
    tokenized_texts = [preprocess_text(t) for t in texts]
    best_topic_num, lda_model, dictionary, corpus = find_optimal_lda(tokenized_texts, start=2, limit=10, step=1, passes=10)
    lda_keywords = extract_lda_keywords(lda_model, top_n=10)
    
    print("\n==== LDA Keyword Extraction (Top 10 per topic) ====")
    for topic_id, keywords in lda_keywords.items():
        print(f"\nTopic {topic_id}:")
        for term, weight in keywords:
            print(f"  {term:15} {weight:.4f}")

    # Opsiyonel: TF-IDF veya LDA sonuçlarını görselleştirmek için grafikler de eklenebilir.


[INFO] CSV'den 1271 yorum okundu.

==== TF-IDF Keyword Extraction (Top 10) ====
room            75.5801
hotel           71.3218
stay            57.5217
clean           55.5063
place           51.9991
nice            44.2168
breakfast       41.8739
location        39.3597
good            36.3183
akyaka          36.1632

[INFO] Optimal topic sayısı aranıyor: 2..10 arası
   2 topics => coherence=0.3604
   3 topics => coherence=0.4039
   4 topics => coherence=0.3936
   5 topics => coherence=0.3905
   6 topics => coherence=0.3770
   7 topics => coherence=0.3763
   8 topics => coherence=0.3644
   9 topics => coherence=0.3624

[INFO] En iyi topic sayısı = 3, coherence = 0.4039

==== LDA Keyword Extraction (Top 10 per topic) ====

Topic 0:
  room            0.0377
  stay            0.0153
  place           0.0114
  clean           0.0114
  hotel           0.0103
  location        0.0089
  pool            0.0084
  night           0.0077
  bed             0.0072
  not             0.0072

Topic 1